# Morocco Crop LightGBM Baseline

This notebook trains a Morocco-only crop-classification baseline from the enriched `features.parquet` file. It is currently configured as a two-class AOI experiment for `Alfalfa` and `Corn` only, with the other labels left commented so they can be restored easily.

The model uses an AOI-compatible brightness-robust 18-feature schema: 9 default Sentinel-2 band means normalized by per-row spectral brightness, plus 9 vegetation-index means. Metadata, coverage, area, weather, geometry, and multi-stat aggregate features are intentionally excluded or left commented.

The notebook saves reusable artifacts and includes a simple AOI prediction check at the end.


## Dependencies

This notebook is intended to run in Colab or a clean notebook environment. LightGBM is the only model family used here; scikit-learn is used only for splitting and evaluation.

In [ ]:
# %pip install -q lightgbm scikit-learn pyarrow


## Load The Enriched Feature Dataset

The previous notebook created `features.parquet`, one enriched time-series table with Sentinel-2 bands, vegetation indices, weather, geometry metadata, and labels. This cell loads that file and supports both Colab-style `data/...` paths and the local workspace copy under `notebooks/data/...`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [
    Path("data/morocco_crop_irrigation"),
    Path("notebooks/data/morocco_crop_irrigation"),
]
DATA_DIR = next(path for path in DATA_DIR_CANDIDATES if (path / "features.parquet").exists())
DATA_PATH = DATA_DIR / "features.parquet"

features_ts = pd.read_parquet(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", features_ts.shape)
print("Unique parcels:", features_ts["parcel_id"].nunique())
print("Columns:")
print(features_ts.columns.tolist())

features_ts.head()


Loaded: data\morocco_crop_irrigation\features.parquet
Shape: (390766, 70)
Unique parcels: 7734
Columns:
['field_id', 'parcel_id', 'satellite_date', 'label', 'model_label', 'survey_date', 'irrigation_type', 'survey_zone', 'country', 'year', 'dataset_source', 'doy', 's2_B01_mean', 's2_B02_mean', 's2_B03_mean', 's2_B04_mean', 's2_B05_mean', 's2_B06_mean', 's2_B07_mean', 's2_B08_mean', 's2_B8A_mean', 's2_B09_mean', 's2_B11_mean', 's2_B12_mean', 's2_NDVI_mean', 's2_SAVI_mean', 's2_EVI_mean', 's2_GNDVI_mean', 's2_NDWI_GREEN_mean', 's2_NDRE_mean', 's2_NDMI_mean', 's2_MSI_mean', 's2_NBR_mean', 's2_BSI_mean', 's2_CAI_B01_B08_mean', 'weather_location_id', 'weather_daily_Temp_mean_C', 'weather_daily_Temp_max_C', 'weather_daily_Temp_min_C', 'weather_daily_Total_prec_mm', 'weather_daily_gdd_base10', 'weather_daily_rainy_day', 'weather_daily_heavy_rain_day', 'weather_daily_heat_stress_day', 'weather_daily_cold_stress_day', 'weather_7d_Total_prec_mm', 'weather_7d_Temp_mean_C', 'weather_7d_gdd_base10'

,field_id,parcel_id,satellite_date,label,model_label,survey_date,irrigation_type,survey_zone,country,year,...,weather_cum_heavy_rain_days,weather_cum_heat_stress_days,weather_cum_cold_stress_days,centroid_lon,centroid_lat,bbox_min_lon,bbox_min_lat,bbox_max_lon,bbox_max_lat,area_ha
0,Doukkala_Bare soil_20240422_001,Doukkala_Bare soil_20240422_001,2024-01-11,Bare soil,Background,2024-04-22,NaN,Doukkala,Morocco,2024,...,0.0,0.0,0.0,-8.733904,32.370678,-8.735481,32.369806,-8.732344,32.371738,2.85092
1,Doukkala_Bare soil_20240422_001,Doukkala_Bare soil_20240422_001,2024-01-21,Bare soil,Background,2024-04-22,NaN,Doukkala,Morocco,2024,...,1.0,0.0,0.0,-8.733904,32.370678,-8.735481,32.369806,-8.732344,32.371738,2.85092
2,Doukkala_Bare soil_20240422_001,Doukkala_Bare soil_20240422_001,2024-01-24,Bare soil,Background,2024-04-22,NaN,Doukkala,Morocco,2024,...,1.0,0.0,0.0,-8.733904,32.370678,-8.735481,32.369806,-8.732344,32.371738,2.85092
3,Doukkala_Bare soil_20240422_001,Doukkala_Bare soil_20240422_001,2024-02-18,Bare soil,Background,2024-04-22,NaN,Doukkala,Morocco,2024,...,1.0,0.0,0.0,-8.733904,32.370678,-8.735481,32.369806,-8.732344,32.371738,2.85092
4,Doukkala_Bare soil_20240422_001,Doukkala_Bare soil_20240422_001,2024-02-20,Bare soil,Background,2024-04-22,NaN,Doukkala,Morocco,2024,...,1.0,0.0,0.0,-8.733904,32.370678,-8.735481,32.369806,-8.732344,32.371738,2.85092


## Clean Duplicate Dates And Unstable EVI

Some source rows contain multiple observations for the same parcel on the same satellite date. This cell averages numeric features for each `parcel_id + satellite_date` pair, keeps stable metadata from the first row, and clips extreme EVI values caused by near-zero formula denominators.


In [ ]:
model_ts = features_ts.copy()
model_ts["satellite_date"] = pd.to_datetime(model_ts["satellite_date"], errors="coerce")

rows_before = len(model_ts)
duplicate_parcel_dates = int(model_ts.duplicated(["parcel_id", "satellite_date"]).sum())

if "s2_EVI_mean" in model_ts.columns:
    evi_before = model_ts["s2_EVI_mean"].describe(percentiles=[0.01, 0.5, 0.99])
    model_ts["s2_EVI_mean"] = model_ts["s2_EVI_mean"].clip(-1.0, 1.0)
    evi_after = model_ts["s2_EVI_mean"].describe(percentiles=[0.01, 0.5, 0.99])
else:
    evi_before = None
    evi_after = None

numeric_columns = model_ts.select_dtypes(include=[np.number]).columns.tolist()
metadata_columns = [col for col in model_ts.columns if col not in numeric_columns]

aggregation_spec = {col: "mean" for col in numeric_columns}
aggregation_spec.update({col: "first" for col in metadata_columns if col not in {"parcel_id", "satellite_date"}})

model_ts = (
    model_ts
    .sort_values(["parcel_id", "satellite_date"])
    .groupby(["parcel_id", "satellite_date"], as_index=False)
    .agg(aggregation_spec)
)

print("Rows before parcel-date cleanup:", rows_before)
print("Duplicate parcel-date rows removed:", duplicate_parcel_dates)
print("Rows after parcel-date cleanup:", len(model_ts))
print("Unique parcels:", model_ts["parcel_id"].nunique())

if evi_before is not None:
    print("EVI before clipping:")
    display(evi_before.to_frame("s2_EVI_mean"))
    print("EVI after clipping:")
    display(evi_after.to_frame("s2_EVI_mean"))

print("Model labels:")
display(model_ts.drop_duplicates("parcel_id")["model_label"].value_counts().to_frame("parcel_count"))


Rows before parcel-date cleanup: 390766
Duplicate parcel-date rows removed: 80852
Rows after parcel-date cleanup: 309914
Unique parcels: 7734
EVI before clipping:


,s2_EVI_mean
count,389618.000000
mean,0.235772
std,0.227387
min,-27.458071
1%,0.055429
50%,0.129759
99%,0.808571
max,38.113894


EVI after clipping:


,s2_EVI_mean
count,389618.000000
mean,0.235802
std,0.206072
min,-1.000000
1%,0.055429
50%,0.129759
99%,0.808571
max,1.000000


Model labels:


,parcel_count
model_label,
Wheat,2614
Other crop,1707
Background,1304
Corn,1252
Alfalfa,857


## Aggregate Time Series To One Row Per Parcel

LightGBM needs a fixed-width table. This cell converts each parcel's cleaned time series into one row, filters active classes through an easy-to-edit label list, and creates the current 18-feature model schema: brightness-normalized default Sentinel-2 band means plus vegetation-index means. Raw bands are retained only as intermediate values for brightness normalization.


In [ ]:
S2_BAND_FEATURES = [
    "s2_B02_mean", "s2_B03_mean", "s2_B04_mean",
    "s2_B05_mean", "s2_B06_mean", "s2_B07_mean",
    "s2_B08_mean", "s2_B8A_mean", "s2_B11_mean",
]

VI_FEATURES = [
    "s2_NDVI_mean", "s2_SAVI_mean", "s2_EVI_mean", "s2_GNDVI_mean",
    "s2_NDWI_GREEN_mean", "s2_NDRE_mean", "s2_NDMI_mean", "s2_MSI_mean",
    "s2_BSI_mean",
]

NORMALIZED_S2_BAND_FEATURES = [f"{col}_brightness_norm" for col in S2_BAND_FEATURES]
MODEL_BASE_FEATURES = NORMALIZED_S2_BAND_FEATURES + VI_FEATURES

INCLUDED_MODEL_LABELS = [
    "Alfalfa",
    "Corn",
    # "Wheat",
    # "Other crop",
    # "Background",
]

# Deliberately not used in this AOI-compatible baseline:
# RAW_S2_BAND_FEATURES = S2_BAND_FEATURES
# WEATHER_FEATURES = [col for col in model_ts.columns if col.startswith("weather_")]
# GEOMETRY_FEATURES = ["centroid_lon", "centroid_lat", "bbox_min_lon", "bbox_min_lat", "bbox_max_lon", "bbox_max_lat", "area_ha"]
# COVERAGE_FEATURES = ["observation_count", "time_span_days"]
# NON_DEFAULT_S2_FEATURES = ["s2_B01_mean", "s2_B09_mean", "s2_B12_mean", "s2_NBR_mean", "s2_CAI_B01_B08_mean"]
# MULTI_STAT_AGGREGATES = ["median", "std", "min", "max"]

required_source_features = S2_BAND_FEATURES + VI_FEATURES
missing_features = [col for col in required_source_features if col not in model_ts.columns]
if missing_features:
    raise ValueError(f"Missing required AOI-compatible source features: {missing_features}")

model_ts = model_ts.sort_values(["parcel_id", "satellite_date"])
features = model_ts.groupby("parcel_id")[required_source_features].mean().reset_index()

brightness = features[S2_BAND_FEATURES].mean(axis=1).replace(0, np.nan)
for source_col, norm_col in zip(S2_BAND_FEATURES, NORMALIZED_S2_BAND_FEATURES, strict=True):
    features[norm_col] = features[source_col] / brightness
features[NORMALIZED_S2_BAND_FEATURES] = features[NORMALIZED_S2_BAND_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)

metadata_columns = ["parcel_id", "model_label", "label", "irrigation_type", "survey_zone"]
metadata = model_ts[metadata_columns].drop_duplicates("parcel_id")
model_table = metadata.merge(features, on="parcel_id", how="inner")

excluded_rows = model_table.loc[~model_table["model_label"].isin(INCLUDED_MODEL_LABELS), ["model_label", "label", "survey_zone"]]
excluded_summary = excluded_rows.value_counts().rename("rows").reset_index()
model_table = model_table.loc[model_table["model_label"].isin(INCLUDED_MODEL_LABELS)].reset_index(drop=True)

print("Included model labels:", INCLUDED_MODEL_LABELS)
print("Excluded rows summary:")
print(excluded_summary.to_string(index=False) if len(excluded_summary) else "None")
print("Raw S2 band features used only for brightness normalization:", len(S2_BAND_FEATURES))
print("Brightness-normalized S2 band model features:", len(NORMALIZED_S2_BAND_FEATURES))
print("Vegetation index model features:", len(VI_FEATURES))
print("Model feature columns:", len(MODEL_BASE_FEATURES))
print("Model table shape:", model_table.shape)
print("Unique parcels:", model_table["parcel_id"].nunique())
print("Excluded from model: raw S2 bands, area_ha, observation_count, time_span_days, weather, geometry, non-default S2 bands, multi-stat aggregates")
print("Model label distribution:")
display(model_table["model_label"].value_counts().to_frame("parcel_count"))

model_table.head()


Included model labels: ['Alfalfa', 'Corn']
Excluded rows summary:
model_label      label survey_zone  rows
      Wheat      Wheat    Doukkala  1447
      Wheat      Wheat       Tadla   765
 Background  Bare soil    Doukkala   347
 Background       Weed    Doukkala   346
      Wheat      Wheat    el haouz   331
 Other crop      Beets    Doukkala   303
 Other crop   Potatoes    Doukkala   287
 Background  Bare soil    el haouz   231
 Other crop     Peanut       Gharb   166
 Other crop       Peas       Gharb   160
 Background  Bare soil      Statte   128
 Other crop   Potatoes    el haouz   119
 Background  Bare soil       Tadla   108
 Other crop  Sugarcane       Gharb    94
 Other crop      Beets       Tadla    94
 Other crop   Tomatoes    Doukkala    75
 Other crop     Onions    el haouz    71
 Background  Bare soil       Souss    70
 Background  Bare soil       Gharb    58
      Wheat      Wheat      Statte    57
 Other crop     Onions       Tadla    52
 Other crop   Tomatoes       Gha

,parcel_count
model_label,
Corn,1252
Alfalfa,857


,parcel_id,model_label,label,irrigation_type,survey_zone,s2_B02_mean,s2_B03_mean,s2_B04_mean,s2_B05_mean,s2_B06_mean,...,s2_BSI_mean,s2_B02_mean_brightness_norm,s2_B03_mean_brightness_norm,s2_B04_mean_brightness_norm,s2_B05_mean_brightness_norm,s2_B06_mean_brightness_norm,s2_B07_mean_brightness_norm,s2_B08_mean_brightness_norm,s2_B8A_mean_brightness_norm,s2_B11_mean_brightness_norm
0,Doukkala_Seasonal Crop_20240422_009,Corn,Corn,Flood Irrigation,Doukkala,0.071996,0.090004,0.118866,0.134542,0.143057,...,0.305742,0.463255,0.579126,0.764839,0.865700,0.920492,1.006112,1.092086,1.156768,2.151621
1,Doukkala_Seasonal Crop_20240422_018,Corn,Corn,Rainfed,Doukkala,0.060391,0.081074,0.113902,0.129904,0.140829,...,0.317531,0.405202,0.543971,0.764238,0.871603,0.944907,1.032511,1.112625,1.175332,2.149611
2,Doukkala_Seasonal Crop_20240422_036,Corn,Corn,Rainfed,Doukkala,0.051380,0.071536,0.105089,0.120424,0.129095,...,0.307460,0.384896,0.535893,0.787249,0.902125,0.967083,1.054297,1.125131,1.191175,2.052151
3,Doukkala_Seasonal Crop_20240422_040,Corn,Corn,Rainfed,Doukkala,0.057134,0.075336,0.103312,0.117904,0.126847,...,0.323413,0.417192,0.550105,0.754387,0.860935,0.926241,1.015276,1.103202,1.170433,2.202229
4,Doukkala_Seasonal Crop_20240422_042,Corn,Corn,Rainfed,Doukkala,0.057762,0.079615,0.113046,0.131127,0.141357,...,0.327071,0.387825,0.534551,0.759010,0.880408,0.949093,1.038453,1.103282,1.184581,2.162797


## Train/Test Split

This baseline uses a stratified train/test split so both active classes are represented in both sets. The current experiment includes only `Alfalfa` and `Corn`, and uses exactly 18 AOI-compatible brightness-robust features: 9 brightness-normalized S2 bands plus 9 vegetation indices. No scaler or normalizer is used because LightGBM is tree-based; the brightness normalization is explicit feature engineering, not a generic scaler.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

feature_columns = MODEL_BASE_FEATURES.copy()

X = model_table[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_table["model_label"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X,
    y_encoded,
    model_table.index,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

print("Feature count:", len(feature_columns))
print("Feature columns:", feature_columns)
print("Classes:", label_encoder.classes_.tolist())
print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Scaler/normalizer: not used (LightGBM tree model)")


Feature count: 18
Feature columns: ['s2_B02_mean_brightness_norm', 's2_B03_mean_brightness_norm', 's2_B04_mean_brightness_norm', 's2_B05_mean_brightness_norm', 's2_B06_mean_brightness_norm', 's2_B07_mean_brightness_norm', 's2_B08_mean_brightness_norm', 's2_B8A_mean_brightness_norm', 's2_B11_mean_brightness_norm', 's2_NDVI_mean', 's2_SAVI_mean', 's2_EVI_mean', 's2_GNDVI_mean', 's2_NDWI_GREEN_mean', 's2_NDRE_mean', 's2_NDMI_mean', 's2_MSI_mean', 's2_BSI_mean']
Classes: ['Alfalfa', 'Corn']
Train rows: 1687
Test rows: 422
Scaler/normalizer: not used (LightGBM tree model)


## Train LightGBM

This is the only model trained in this notebook. Class balancing is enabled. The LightGBM objective is selected automatically: `binary` when two labels are active, `multiclass` when three or more labels are active.


In [ ]:
from lightgbm import LGBMClassifier

lightgbm_objective = "binary" if len(label_encoder.classes_) == 2 else "multiclass"

model = LGBMClassifier(
    objective=lightgbm_objective,
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

model.fit(X_train, y_train)

print("LightGBM training complete.")
print("Objective:", lightgbm_objective)


LightGBM training complete.
Objective: binary


## Evaluate The Baseline

This cell reports overall accuracy, macro F1, per-class precision/recall/F1, and a confusion matrix. Macro F1 is important because it gives minority classes more weight than accuracy alone.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

pred_encoded = model.predict(X_test)
pred_labels = label_encoder.inverse_transform(pred_encoded)
true_labels = label_encoder.inverse_transform(y_test)

accuracy = accuracy_score(true_labels, pred_labels)
macro_f1 = f1_score(true_labels, pred_labels, average="macro")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("\nClassification report:")
print(classification_report(true_labels, pred_labels, digits=4))

confusion = pd.DataFrame(
    confusion_matrix(true_labels, pred_labels, labels=label_encoder.classes_),
    index=[f"actual_{label}" for label in label_encoder.classes_],
    columns=[f"pred_{label}" for label in label_encoder.classes_],
)

display(confusion)

Accuracy: 0.9502369668246445
Macro F1: 0.9486138734423835

Classification report:
              precision    recall  f1-score   support

     Alfalfa     0.9261    0.9532    0.9395       171
        Corn     0.9675    0.9482    0.9577       251

    accuracy                         0.9502       422
   macro avg     0.9468    0.9507    0.9486       422
weighted avg     0.9507    0.9502    0.9503       422



,pred_Alfalfa,pred_Corn
actual_Alfalfa,163,8
actual_Corn,13,238


## Review Errors And Metadata Patterns

This cell inspects wrong predictions and summarizes them by true/predicted class. Irrigation type and survey zone are not model features, but they are useful for diagnosing where errors concentrate.

In [ ]:
test_results = model_table.loc[test_idx, ["parcel_id", "label", "model_label", "irrigation_type", "survey_zone"]].copy()
test_results["predicted_label"] = pred_labels
test_results["is_correct"] = test_results["model_label"] == test_results["predicted_label"]

print("Error count:", (~test_results["is_correct"]).sum())

print("\nErrors by actual/predicted label:")
display(
    test_results.loc[~test_results["is_correct"]]
    .groupby(["model_label", "predicted_label"])
    .size()
    .sort_values(ascending=False)
    .to_frame("count")
)

print("\nAccuracy by survey zone:")
display(test_results.groupby("survey_zone")["is_correct"].mean().sort_values().to_frame("accuracy"))

print("\nAccuracy by irrigation type:")
display(test_results.groupby("irrigation_type", dropna=False)["is_correct"].mean().sort_values().to_frame("accuracy"))

Error count: 21

Errors by actual/predicted label:


,,count
model_label,predicted_label,
Corn,Alfalfa,13
Alfalfa,Corn,8



Accuracy by survey zone:


,accuracy
survey_zone,
Statte,0.800000
el haouz,0.882353
Souss,0.909091
Tadla,0.923077
Gharb,0.956522
Doukkala,0.960000



Accuracy by irrigation type:


,accuracy
irrigation_type,
Basin Irrigation,0.900000
Drip Irrigation,0.946429
Flood Irrigation,0.947977
Sprinkler Irrigation,0.950820
Rainfed,0.966102
Pivot Irrigation,1.000000


## Feature Importance

This cell shows the strongest LightGBM features among the active 18 inputs: brightness-normalized S2 band means and vegetation indices. It helps check whether the binary Alfalfa/Corn model is relying mostly on VIs or normalized red/edge/NIR/SWIR band shape.


In [ ]:
feature_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": model.feature_importances_,
    }
).sort_values("importance", ascending=False)

feature_importance.head(30)

,feature,importance
5,s2_B07_mean_brightness_norm,1992
1,s2_B03_mean_brightness_norm,1706
4,s2_B06_mean_brightness_norm,1691
3,s2_B05_mean_brightness_norm,1623
12,s2_GNDVI_mean,1447
0,s2_B02_mean_brightness_norm,1044
13,s2_NDWI_GREEN_mean,1023
10,s2_SAVI_mean,995
11,s2_EVI_mean,984
7,s2_B8A_mean_brightness_norm,934


## Baseline Checkpoint

This notebook now has a first LightGBM-only baseline for the grouped target: `Wheat`, `Corn`, `Alfalfa`, `Other crop`, and `Background`. The next section adds stricter leave-one-zone-out cross-validation to test geographic generalization.


## Leave-One-Zone-Out Cross-Validation

The random stratified split trains and tests on parcels drawn from all zones, so it measures same-distribution performance. This stricter validation uses scikit-learn's `LeaveOneGroupOut`, with `survey_zone` as the group, so each fold holds out one entire zone as unseen test data.


In [ ]:
print("Parcels by survey zone and model label:")
zone_label_counts = pd.crosstab(model_table["survey_zone"], model_table["model_label"])
display(zone_label_counts)

print("\nTotal parcels by zone:")
display(model_table["survey_zone"].value_counts().to_frame("parcel_count"))

Parcels by survey zone and model label:


model_label,Alfalfa,Corn
survey_zone,,
Doukkala,571,525
Gharb,63,664
Souss,4,48
Statte,17,0
Tadla,116,2
el haouz,86,13



Total parcels by zone:


,parcel_count
survey_zone,
Doukkala,1096
Gharb,727
Tadla,118
el haouz,99
Souss,52
Statte,17


## Train LightGBM With Leave-One-Zone-Out Folds

Each fold below is created by `LeaveOneGroupOut`. The model never sees the held-out zone during training, so the results are a stronger test of geographic generalization than the random split.


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut


def make_lightgbm_model():
    objective = "binary" if len(label_encoder.classes_) == 2 else "multiclass"
    return LGBMClassifier(
        objective=objective,
        n_estimators=600,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.9,
        colsample_bytree=0.9,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

zone_groups = model_table["survey_zone"].fillna("Unknown zone").astype(str)
leave_one_zone_out = LeaveOneGroupOut()

zone_results = []
zone_reports = {}
zone_confusions = {}

for train_positions, test_positions in leave_one_zone_out.split(X, y, groups=zone_groups):
    held_out_zone = zone_groups.iloc[test_positions].iloc[0]

    X_zone_train = X.iloc[train_positions]
    X_zone_test = X.iloc[test_positions]
    y_zone_train = y.iloc[train_positions]
    y_zone_test = y.iloc[test_positions]

    zone_model = make_lightgbm_model()
    zone_model.fit(X_zone_train, y_zone_train)

    zone_pred_labels = zone_model.predict(X_zone_test)

    zone_accuracy = accuracy_score(y_zone_test, zone_pred_labels)
    zone_macro_f1 = f1_score(
        y_zone_test,
        zone_pred_labels,
        labels=label_encoder.classes_,
        average="macro",
        zero_division=0,
    )

    zone_results.append(
        {
            "held_out_zone": held_out_zone,
            "train_parcels": int(len(train_positions)),
            "test_parcels": int(len(test_positions)),
            "accuracy": zone_accuracy,
            "macro_f1": zone_macro_f1,
        }
    )

    zone_reports[held_out_zone] = classification_report(
        y_zone_test,
        zone_pred_labels,
        labels=label_encoder.classes_,
        zero_division=0,
        output_dict=True,
    )

    zone_confusions[held_out_zone] = pd.DataFrame(
        confusion_matrix(y_zone_test, zone_pred_labels, labels=label_encoder.classes_),
        index=[f"actual_{label}" for label in label_encoder.classes_],
        columns=[f"pred_{label}" for label in label_encoder.classes_],
    )

zone_results = pd.DataFrame(zone_results).sort_values("macro_f1")
display(zone_results)



,held_out_zone,train_parcels,test_parcels,accuracy,macro_f1
1,Gharb,1382,727,0.477304,0.421934
3,Statte,2092,17,0.823529,0.451613
2,Souss,2057,52,0.673077,0.525497
4,Tadla,1991,118,0.889831,0.537255
0,Doukkala,1013,1096,0.671533,0.650311
5,el haouz,2010,99,0.929293,0.849903


## Inspect Leave-One-Zone-Out Weak Spots

This cell expands the leave-one-zone-out results into per-class F1 scores. It helps identify whether low performance comes from one difficult zone, one difficult class, or both.


In [ ]:
per_zone_class_rows = []

for zone, report in zone_reports.items():
    for label in label_encoder.classes_:
        per_zone_class_rows.append(
            {
                "held_out_zone": zone,
                "class": label,
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1": report[label]["f1-score"],
                "support": report[label]["support"],
            }
        )

per_zone_class_metrics = pd.DataFrame(per_zone_class_rows)

display(
    per_zone_class_metrics
    .pivot(index="held_out_zone", columns="class", values="f1")
    .loc[zone_results["held_out_zone"]]
)

worst_zone = zone_results.iloc[0]["held_out_zone"]
print("Worst held-out zone by macro F1:", worst_zone)
display(zone_confusions[worst_zone])

class,Alfalfa,Corn
held_out_zone,,
Gharb,0.243028,0.600840
Statte,0.903226,0.000000
Souss,0.260870,0.790123
Tadla,0.941176,0.133333
Doukkala,0.736457,0.564165
el haouz,0.959064,0.740741


Worst held-out zone by macro F1: Gharb


,pred_Alfalfa,pred_Corn
actual_Alfalfa,61,2
actual_Corn,378,286


## Validation Interpretation

Compare the random stratified split with the leave-one-zone-out cross-validation results. If performance drops sharply when a full zone is unseen during training, the model is learning patterns that do not transfer equally across Moroccan survey zones. In that case, the next improvement should focus on validation design and feature robustness before claiming general crop generalization.


## Save The Morocco Baseline Model

Retrain the selected LightGBM configuration on all Morocco parcels and save reusable artifacts. The output path is relative to this notebook, so running from `notebooks/` writes to `notebooks/data/models/morocco_crop_lightgbm_baseline/`.

In [ ]:
import json
from datetime import datetime, timezone

import joblib

final_model = make_lightgbm_model()
final_model.fit(X, y_encoded)

artifact_dir = Path("data/models/morocco_crop_lightgbm_baseline")
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "model.pkl"
encoder_path = artifact_dir / "label_encoder.pkl"
features_path = artifact_dir / "feature_columns.pkl"
metadata_path = artifact_dir / "metadata.json"

joblib.dump(final_model, model_path)
joblib.dump(label_encoder, encoder_path)
joblib.dump(feature_columns, features_path)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_family": "LightGBM",
    "dataset": "morocco_crop_irrigation/features.parquet",
    "rows": int(len(X)),
    "classes": label_encoder.classes_.tolist(),
    "included_model_labels": INCLUDED_MODEL_LABELS,
    "feature_count": len(feature_columns),
    "feature_columns": feature_columns,
    "feature_schema": "9 brightness-normalized default S2 band means + 9 VI means",
    "brightness_normalization": "Each S2 band mean divided by the row mean of B02/B03/B04/B05/B06/B07/B08/B8A/B11.",
    "excluded_features": [
        "raw S2 band means as direct model inputs",
        "area_ha",
        "observation_count",
        "time_span_days",
        "weather_*",
        "geometry/coordinate columns",
        "non-default S2 bands B01/B09/B12 and dependent indices",
        "multi-stat aggregates median/std/min/max",
    ],
    "validation_accuracy": float(accuracy),
    "validation_macro_f1": float(macro_f1),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved model:", model_path)
print("Saved label encoder:", encoder_path)
print("Saved feature columns:", features_path)
print("Saved metadata:", metadata_path)
print("Artifact summary:")
print("  Classes:", label_encoder.classes_.tolist())
print("  Features:", len(feature_columns))
print("  Feature schema:", metadata["feature_schema"])


Saved model: data\models\morocco_crop_lightgbm_baseline\model.pkl
Saved label encoder: data\models\morocco_crop_lightgbm_baseline\label_encoder.pkl
Saved feature columns: data\models\morocco_crop_lightgbm_baseline\feature_columns.pkl
Saved metadata: data\models\morocco_crop_lightgbm_baseline\metadata.json
Artifact summary:
  Classes: ['Alfalfa', 'Corn']
  Features: 18
  Feature schema: 9 brightness-normalized default S2 band means + 9 VI means


## Simple AOI Prediction Check

This cell loads the saved Morocco-only model and predicts known AOI seasons directly. It is intentionally straightforward: no assertions, no pytest, and no generated CSVs.

In [ ]:
import xarray as xr

AOI_DIR_CANDIDATES = [
    Path("../data/aoi_demo_01"),
    Path("data/aoi_demo_01"),
    Path(r"C:\Users\abdulhamed\Desktop\work\FarmTrust\data\aoi_demo_01"),
]
AOI_DIR = next(path for path in AOI_DIR_CANDIDATES if path.exists())

MODEL_DIR_CANDIDATES = [
    Path("data/models/morocco_crop_lightgbm_baseline"),
    Path("notebooks/data/models/morocco_crop_lightgbm_baseline"),
]
MODEL_DIR = next(path for path in MODEL_DIR_CANDIDATES if (path / "model.pkl").exists())

CUBE_PATH = AOI_DIR / "cube.zarr"
INDICES_PATH = AOI_DIR / "indices_timeseries.csv"
VALID_SCL = {2, 4, 5, 6}
AOI_SEASONS = {
    "C1": {"label": "Corn", "start": "2024-05-18", "end": "2024-08-30"},
    "C3": {"label": "Corn", "start": "2025-06-15", "end": "2025-09-11"},
    "C4": {"label": "Alfalfa", "start": "2025-10-05", "end": "2026-05-31"},
}
AOI_BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11"]

saved_model = joblib.load(MODEL_DIR / "model.pkl")
saved_label_encoder = joblib.load(MODEL_DIR / "label_encoder.pkl")
saved_feature_columns = joblib.load(MODEL_DIR / "feature_columns.pkl")

indices = pd.read_csv(INDICES_PATH, parse_dates=["solar_day"])
ds_10m = xr.open_zarr(CUBE_PATH, consolidated=False)
ds_20m = xr.open_zarr(CUBE_PATH, group="20m", consolidated=False)
scl_10m = ds_20m["SCL"].sel(y=ds_10m.y, x=ds_10m.x, method="nearest")
time_idx = {pd.Timestamp(t).date(): i for i, t in enumerate(ds_10m.time.values)}


def safe_div(num: float, den: float) -> float:
    return float("nan") if abs(den) < 1e-10 else num / den


def compute_vi_from_reflectance(bands: dict[str, float]) -> dict[str, float]:
    b02 = bands["B02"]
    b03 = bands["B03"]
    b04 = bands["B04"]
    b05 = bands["B05"]
    b08 = bands["B08"]
    b11 = bands["B11"]
    return {
        "s2_SAVI_mean": safe_div((b08 - b04) * 1.5, b08 + b04 + 0.5),
        "s2_GNDVI_mean": safe_div(b08 - b03, b08 + b03),
        "s2_NDRE_mean": safe_div(b08 - b05, b08 + b05),
        "s2_MSI_mean": safe_div(b11, b08),
        "s2_BSI_mean": safe_div((b11 + b04) - (b08 + b02), (b11 + b04) + (b08 + b02)),
    }


def season_features(start: str, end: str) -> pd.DataFrame:
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    dates_in_data = indices[
        (indices["solar_day"] >= start_ts)
        & (indices["solar_day"] <= end_ts)
        & (indices["valid_fraction"] > 0)
    ]

    per_date_bands = {band: [] for band in AOI_BANDS}
    computed_vis = {key: [] for key in ["s2_SAVI_mean", "s2_GNDVI_mean", "s2_NDRE_mean", "s2_MSI_mean", "s2_BSI_mean"]}

    for dt in dates_in_data["solar_day"]:
        dt_date = pd.Timestamp(dt).date()
        if dt_date not in time_idx:
            continue
        ti = time_idx[dt_date]
        valid_mask = np.isin(scl_10m.isel(time=ti).values, list(VALID_SCL))
        valid_count = int(valid_mask.sum())
        if valid_count == 0:
            continue

        band_values = {}
        for band in AOI_BANDS:
            if band in ds_10m:
                arr = ds_10m[band].isel(time=ti).values.astype(float)
            else:
                arr_20m = ds_20m[band].isel(time=ti).values.astype(float)
                arr = xr.DataArray(arr_20m, dims=["y", "x"], coords={"y": ds_20m.y, "x": ds_20m.x}).sel(
                    y=ds_10m.y, x=ds_10m.x, method="nearest"
                ).values
            reflectance = (arr * valid_mask).sum() / valid_count / 10000.0
            band_values[band] = reflectance
            per_date_bands[band].append(reflectance)

        vi_values = compute_vi_from_reflectance(band_values)
        for key, value in vi_values.items():
            computed_vis[key].append(value)

    features = {}
    for band in AOI_BANDS:
        features[f"s2_{band}_mean"] = np.nanmean(per_date_bands[band])

    csv_vis = {
        "s2_NDVI_mean": "ndvi_mean",
        "s2_EVI_mean": "evi_mean",
        "s2_NDMI_mean": "ndmi_mean",
        "s2_NDWI_GREEN_mean": "ndwi_mean",
    }
    for feature_name, csv_col in csv_vis.items():
        features[feature_name] = dates_in_data[csv_col].mean()
    for feature_name, values in computed_vis.items():
        features[feature_name] = np.nanmean(values)

    brightness_source_cols = [f"s2_{band}_mean" for band in AOI_BANDS]
    brightness = np.nanmean([features[col] for col in brightness_source_cols])
    for col in brightness_source_cols:
        features[f"{col}_brightness_norm"] = features[col] / brightness if brightness and not np.isnan(brightness) else np.nan

    return pd.DataFrame([features]).reindex(columns=saved_feature_columns).replace([np.inf, -np.inf], np.nan).fillna(0)


prediction_rows = []
for season_name, season in AOI_SEASONS.items():
    X_aoi = season_features(season["start"], season["end"])
    pred_encoded = saved_model.predict(X_aoi)[0]
    pred_label = saved_label_encoder.inverse_transform([pred_encoded])[0]
    probabilities = saved_model.predict_proba(X_aoi)[0]
    row = {
        "season": season_name,
        "known": season["label"],
        "predicted": pred_label,
    }
    row.update({f"prob_{label}": prob for label, prob in zip(saved_label_encoder.classes_, probabilities, strict=True)})
    prediction_rows.append(row)

aoi_predictions = pd.DataFrame(prediction_rows)
print("AOI directory:", AOI_DIR)
print("Model directory:", MODEL_DIR)
print("Saved feature columns:", saved_feature_columns)
display(aoi_predictions)


AOI directory: ..\data\aoi_demo_01
Model directory: data\models\morocco_crop_lightgbm_baseline
Saved feature columns: ['s2_B02_mean_brightness_norm', 's2_B03_mean_brightness_norm', 's2_B04_mean_brightness_norm', 's2_B05_mean_brightness_norm', 's2_B06_mean_brightness_norm', 's2_B07_mean_brightness_norm', 's2_B08_mean_brightness_norm', 's2_B8A_mean_brightness_norm', 's2_B11_mean_brightness_norm', 's2_NDVI_mean', 's2_SAVI_mean', 's2_EVI_mean', 's2_GNDVI_mean', 's2_NDWI_GREEN_mean', 's2_NDRE_mean', 's2_NDMI_mean', 's2_MSI_mean', 's2_BSI_mean']


,season,known,predicted,prob_Alfalfa,prob_Corn
0,C1,Corn,Corn,0.000153,0.999847
1,C3,Corn,Corn,0.008052,0.991948
2,C4,Alfalfa,Alfalfa,0.568402,0.431598
